<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/Practica_2_RAG_%D0%B0%D1%81%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%82_%D0%BF%D0%BE_%D0%BF%D1%81%D0%B8%D1%85%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# RAG-ассистент по локальным документам — полное руководство

## Что мы делаем

Строим RAG (Retrieval-Augmented Generation): программа сначала ищет в вашем документе фрагменты, семантически близкие к вопросу, а потом просит локальную чат-модель ответить, опираясь на них. Всё работает офлайн, через Ollama.

**Что нужно:**
- Ollama установлена и запущена (значок в трее);
- Python 3.10 или новее;
- Доступ в интернет для скачивания моделей через ModelScope (официальный реестр `registry.ollama.ai` в вашей сети блокируется — обходим через зеркало).

---

## Шаг 1. Создайте папку проекта

Все файлы лежат **в одной папке**, откуда вы запускаете скрипты. Например, `C:\Users\student\Downloads\13232\`:

```
13232/
├── documents.txt      ← исходный текст (создаём на шаге 2)
├── build_index.py     ← скрипт индексации (шаг 4)
├── rag_chat.py        ← RAG-чат (шаг 5)
├── chunks.json        ← создаётся build_index.py
└── embeddings.npy     ← создаётся build_index.py
```

Откройте эту папку в VS Code: **File → Open Folder…** → выберите `13232`.

---

## Шаг 2. Создайте `documents.txt`

Документ — обычный текст в **UTF-8**. Размер — 3–10 страниц (10–20 абзацев). Чем чётче тема и чем больше разных подтем, тем лучше работает поиск: с 2 чанками любая ерунда даёт сходство 0.4–0.5, а с 15 чанками релевантные вопросы — 0.6–0.8, нерелевантные — 0.1–0.2.

Ниже — **готовый документ про когнитивные искажения**, проверенный на этой конфигурации. Скопируйте его целиком в файл `documents.txt` в папке проекта. Можно заменить на любой свой текст на одну тему (психология, конспект лекций, глава учебника — не важно, главное связность).

```text
# Когнитивные искажения: краткий обзор

## Что такое когнитивные искажения

Когнитивные искажения — это систематические ошибки в мышлении, которые влияют на наши суждения и решения. Они возникают потому, что мозг использует эвристики — быстрые упрощённые правила обработки информации. В большинстве случаев эвристики полезны: они экономят время и позволяют быстро реагировать. Но иногда приводят к предсказуемым ошибкам, которые повторяются у разных людей в одинаковых ситуациях. Когнитивные искажения не являются признаком глупости или психического расстройства — это универсальная особенность человеческого мышления.

## Катастрофизация

Катастрофизация — это искажение, при котором человек преувеличивает вероятность и серьёзность негативных последствий. Студент может думать: «Если я не сдам этот экзамен, вся моя жизнь рухнет». В реальности один экзамен редко определяет всю биографию. Катастрофизация часто встречается при тревожных расстройствах и проявляется в мыслях, начинающихся со слов «а что если…» и заканчивающихся самым страшным сценарием. Работа с этим искажением включает проверку фактов: сколько раз в прошлом худший сценарий действительно реализовывался?

## Сверхобобщение

Сверхобобщение — это вывод общего правила на основе одного или нескольких случаев. Человек, получивший отказ на собеседовании, может решить: «Меня никогда никуда не возьмут». Сверхобобщение легко узнать по словам-маркерам «всегда», «никогда», «все», «никто», «каждый раз». Один неудачный опыт не описывает закономерность. Полезный приём — заменить абсолютные формулировки на вероятностные: не «я всегда опаздываю», а «сегодня я опоздал».

## Чёрно-белое мышление

Чёрно-белое мышление, или мышление «всё или ничего», — склонность видеть ситуацию только в двух категориях, без промежуточных вариантов. Например: «Если я не лучший, значит я неудачник» или «Если я не согласен полностью, значит я против». Такое мышление не учитывает спектр возможных результатов и промежуточных степеней успеха. Часто встречается у перфекционистов.

## Чтение мыслей

Чтение мыслей — уверенность в том, что вы знаете, о чём думают другие, без достаточных доказательств. Человек может решить: «Он не поздоровался, значит, он меня ненавидит», хотя у собеседника мог быть просто плохой день или он вас не заметил. Это искажение особенно характерно для социальной тревожности. Контрприём — задать себе вопрос: «Какие есть другие объяснения его поведения?»

## Персонализация

Персонализация — склонность приписывать себе ответственность за события, которые произошли не по вашей вине или зависели от множества факторов. Например, родитель может винить себя за плохое поведение ребёнка в школе, хотя на это влияют десятки причин: обстановка в классе, отношения с учителем, особенности характера ребёнка. Персонализация усиливает чувство вины и может приводить к снижению самооценки.

## Фильтрация (избирательное внимание)

Фильтрация — фокусировка на негативных деталях при игнорировании позитивных. Получив отзыв с девятью положительными пунктами и одним замечанием, человек концентрируется только на замечании и считает весь отзыв провальным. В английской литературе это искажение называют «mental filter» или «disqualifying the positive». Работа с фильтрацией включает намеренное внимание к тому, что было сделано хорошо.

## Долженствование

Долженствование — это жёсткие убеждения о том, как «должны» быть устроены мир, люди и сам человек. Формулировки «я должен», «они обязаны», «так не должно быть» создают внутреннее напряжение, когда реальность не совпадает с ожиданиями. Долженствование часто лежит в основе гнева (когда другие нарушают наши правила) и чувства вины (когда мы сами не соответствуем своим же требованиям). Гибкая замена — «мне хотелось бы» вместо «я должен».

## Эффект подтверждения

Эффект подтверждения (confirmation bias) — тенденция искать, интерпретировать и запоминать информацию, которая подтверждает уже существующие убеждения, и игнорировать противоречащую. Человек, уверенный, что «все водители марки X — лихачи», будет замечать только такие случаи, а спокойных водителей не запомнит. Это одно из самых изученных искажений в психологии; оно влияет на политические взгляды, медицинские решения и научные гипотезы.

## Эвристика доступности

Эвристика доступности — склонность оценивать вероятность события по тому, насколько легко вспомнить пример. После новостей об авиакатастрофе люди переоценивают риск полёта, хотя статистически он ниже, чем риск поездки на автомобиле. Эвристика доступности работает через память: чем ярче и эмоциональнее воспоминание, тем выше кажется вероятность. Её описал Даниэль Канеман в книге «Думай медленно… решай быстро».

## Эффект якоря

Эффект якоря — тенденция опираться на первое полученное число или факт при принятии решений. Если в магазине сначала показать цену 10 000 рублей, а потом 5 000, вторая цена кажется выгодной, даже если рыночная стоимость товара — 3 000. Первое число становится «якорем», от которого человек отталкивается, даже если осознаёт, что оно произвольное. Эффект якоря используют в переговорах, продажах и судебных процессах.

## Ошибка выжившего

Ошибка выжившего — искажение, при котором выводы делаются только по «успешным» случаям, а неудачные выпадают из рассмотрения. Классический пример: «Все известные предприниматели бросили университет, значит, образование не нужно». При этом не учитываются тысячи бросивших, которые не добились успеха. Термин возник в исследованиях Второй мировой войны: при анализе повреждений самолётов, вернувшихся с задания, инженеры предлагали укреплять те места, куда попадали пули. Статистик Абрахам Вальд указал, что укреплять нужно, наоборот, места без повреждений — самолёты с повреждениями в этих местах просто не возвращались.

## Как работать с когнитивными искажениями

Первое — научиться их замечать. Полезно вести дневник мыслей: записывать автоматические мысли в стрессовых ситуациях и проверять, какое искажение в них присутствует. Второе — формулировать альтернативные объяснения. Если пришла мысль «все меня считают скучным», спросите себя: какие есть доказательства? какие есть контрпримеры? Третье — при сильной тревоге, депрессии или постоянном стрессе стоит обратиться к психотерапевту. Когнитивно-поведенческая терапия (КПТ) специально работает с когнитивными искажениями и считается одним из наиболее исследованных подходов.

## Заключение

Когнитивные искажения — не признак глупости или психического расстройства, а универсальная особенность человеческого мышления. Все люди в той или иной степени подвержены катастрофизации, сверхобобщению, эффекту подтверждения и другим искажениям. Знание своих типичных искажений помогает принимать более взвешенные решения и меньше страдать от тревожных мыслей. Осознанность — первый шаг к изменениям.
```

Сохраните файл в кодировке UTF-8. В VS Code: правый нижний угол → выберите `UTF-8`. Проверьте, что текст отображается нормально (не «кракозябрами»).

> **Если хотите свою тему** — замените текст, но сохраните структуру: 10–15 подразделов по 2–4 абзаца, каждый про отдельное понятие. Это ключ к тому, что разные вопросы попадают в разные чанки.

---

## Шаг 3. Скачайте модели и создайте псевдонимы

### 3.1. Модель эмбеддингов

```powershell
ollama pull modelscope.cn/nomic-ai/nomic-embed-text-v1.5-GGUF
```

Успех: вывод заканчивается `success`, размер ~84 МБ. Это квантованная GGUF-версия `nomic-embed-text-v1.5` — та же модель, что и оригинальные 274 МБ, но в 3 раза компактнее.

### 3.2. Чат-модель

```powershell
ollama pull modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF
```

Размер ~2.1 ГБ. Скачивание занимает 1–3 минуты в зависимости от скорости.

### 3.3. Псевдонимы (обязательно)

Полные имена `modelscope.cn/...` неудобно писать в коде. Псевдонимы ссылаются на те же файлы на диске — второго экземпляра модели не создаётся.

```powershell
ollama cp modelscope.cn/nomic-ai/nomic-embed-text-v1.5-GGUF:latest nomic-embed-text
ollama cp modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest qwen2.5:3b
```

Каждая команда должна вывести:

```
copied 'modelscope.cn/...' to 'nomic-embed-text'
success
```

### 3.4. Проверка

```powershell
ollama list
```

В списке должно быть **четыре** строки:

```
NAME                                                        ID              SIZE      MODIFIED
nomic-embed-text:latest                                     ...             84 MB     ...
modelscope.cn/nomic-ai/nomic-embed-text-v1.5-GGUF:latest    ...             84 MB     ...
modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest          ...             2.1 GB    ...
qwen2.5:3b                                                  ...             2.1 GB    ...
```

Если псевдонима нет — повторите `ollama cp` с точным именем из списка. Опечатка в одну букву — и модель не создастся.

### 3.5. Установите `numpy`

```powershell
pip install numpy
```

**Если pip падает** с `Failed to establish a new connection: [WinError 10061]` — в системе остались переменные прокси (`HTTP_PROXY`/`HTTPS_PROXY`), указывающие на неработающий прокси. Отключите их **для текущей сессии терминала**:

```powershell
$env:HTTP_PROXY = ""
$env:HTTPS_PROXY = ""
pip install numpy
```

Если прокси нужен для Ollama, но не для pip — используйте флаг только для установки:

```powershell
pip install --proxy "" numpy
```

---

## Шаг 4. `build_index.py` — построение индекса

Этот скрипт читает `documents.txt`, режет его на перекрывающиеся куски по 600 символов, считает для каждого эмбеддинг через `/api/embed` и сохраняет результаты в `chunks.json` и `embeddings.npy`. Запускается один раз (или заново, если поменяли текст).

Создайте файл `build_index.py` в папке проекта:

```python
import json
import os
import requests
import numpy as np

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
EMBED_URL = "http://localhost:11434/api/embed"
EMBED_MODEL = "nomic-embed-text"    # псевдоним, созданный через ollama cp

CHUNK_SIZE = 600                    # символов в одном чанке
OVERLAP = 120                       # перекрытие между соседними чанками

DOCS_PATH = "documents.txt"
OUT_CHUNKS = "chunks.json"
OUT_EMBEDDINGS = "embeddings.npy"


def read_documents(filename=DOCS_PATH):
    """Читает текст из файла и возвращает одну большую строку."""
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"Файл с документами не найден: {filename}\n"
            f"Создайте его рядом со скриптом (см. Шаг 2 руководства)."
        )
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()


def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Режет текст на перекрывающиеся куски."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:                       # пропускаем пустые куски
            chunks.append(chunk)
        start = end - overlap           # следующий кусок с перекрытием
    return chunks


def get_embeddings_batch(texts):
    """Отправляет список текстов в Ollama, возвращает список векторов."""
    payload = {"model": EMBED_MODEL, "input": texts}
    response = requests.post(EMBED_URL, json=payload, timeout=300)
    response.raise_for_status()
    data = response.json()
    if "embeddings" not in data:
        raise RuntimeError(
            f"Неожиданный ответ от Ollama: {data}\n"
            f"Проверьте, что модель '{EMBED_MODEL}' загружена: ollama list"
        )
    return data["embeddings"]


def build_and_save_index():
    text = read_documents()
    chunks = split_into_chunks(text)
    if not chunks:
        raise RuntimeError("Текст пустой — нечего индексировать.")
    print(f"Получено {len(chunks)} чанков. Считаю эмбеддинги...")

    all_vectors = []
    batch_size = 16
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        print(f"  Батч {i // batch_size + 1}: чанки {i + 1}–{i + len(batch)}...")
        try:
            all_vectors.extend(get_embeddings_batch(batch))
        except requests.exceptions.ConnectionError:
            raise RuntimeError(
                "Ollama не запущена. Запустите приложение Ollama и повторите."
            )

    arr = np.array(all_vectors)
    print(f"Размерность эмбеддингов: {arr.shape}")   # ожидаем (N, 768)

    with open(OUT_CHUNKS, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    np.save(OUT_EMBEDDINGS, arr)
    print(f"Готово: {OUT_CHUNKS} и {OUT_EMBEDDINGS} сохранены.")


if __name__ == "__main__":
    build_and_save_index()
```

**Запуск:**

```powershell
python build_index.py
```

**Ожидаемый вывод** (для документа из шага 2, ~14 абзацев):

```
Получено 12 чанков. Считаю эмбеддинги...
  Батч 1: чанки 1–12...
Размерность эмбеддингов: (12, 768)
Готово: chunks.json и embeddings.npy сохранены.
```

Число **768** — размерность векторов `nomic-embed-text-v1.5`. Если увидите другое — проверьте, что в `EMBED_MODEL` стоит именно `nomic-embed-text`, а не случайно не другая модель.

> **Что важно понимать в коде:** `/api/embed` — актуальный эндпоинт для эмбеддингов. Он принимает массив `input` и возвращает массив `embeddings`. Старый `/api/embeddings` (единственное число) батчи не поддерживает и считается устаревшим. Батчи по 16 штук ускоряют индексацию в 10–15 раз по сравнению с поштучной отправкой.

---

## Шаг 5. `rag_chat.py` — RAG-чат

Загружает индекс, принимает вопрос, находит топ-K ближайших чанков по косинусному сходству и просит модель ответить, опираясь на них.

Создайте файл `rag_chat.py`:

```python
import json
import requests
import numpy as np

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
CHAT_URL = "http://localhost:11434/api/chat"
EMBED_URL = "http://localhost:11434/api/embed"

CHAT_MODEL = "qwen2.5:3b"           # псевдоним, созданный через ollama cp
EMBED_MODEL = "nomic-embed-text"

TOP_K = 3                           # сколько фрагментов подставлять в промпт
TEMPERATURE = 0.4                   # не 0.3: при 0.3 модель чаще «сдаётся»


def load_index():
    """Загружает чанки и их векторы из файлов."""
    with open("chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)
    vectors = np.load("embeddings.npy")
    if vectors.shape[0] != len(chunks):
        raise RuntimeError(
            f"Несовпадение: {vectors.shape[0]} векторов и {len(chunks)} чанков. "
            f"Пересоберите индекс: python build_index.py"
        )
    return chunks, vectors


def get_embedding(text):
    """Считает эмбеддинг для вопроса пользователя."""
    payload = {"model": EMBED_MODEL, "input": text}
    response = requests.post(EMBED_URL, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    if "embeddings" not in data or not data["embeddings"]:
        raise RuntimeError(
            f"Неожиданный ответ от Ollama: {data}\n"
            f"Проверьте модель '{EMBED_MODEL}': ollama list"
        )
    return data["embeddings"][0]


def cosine_similarity(a, b):
    """Косинусное сходство двух векторов: от -1 до 1, чем ближе к 1 — тем похожее."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def find_relevant_chunks(question, chunks, vectors, top_k=TOP_K):
    """Возвращает top_k чанков, наиболее близких к вопросу."""
    q_vec = get_embedding(question)
    scores = [(cosine_similarity(q_vec, v), i) for i, v in enumerate(vectors)]
    scores.sort(reverse=True, key=lambda x: x[0])
    return [(chunks[i], sim) for sim, i in scores[:top_k]]


def build_prompt(question, relevant_chunks):
    """Промпт: инструкция, затем КОНТЕКСТ, затем вопрос в конце.

    Порядок «контекст → вопрос» важен: маленькие модели (3B) лучше
    опираются на фрагменты, когда вопрос стоит последним.
    """
    context_parts = []
    for i, (chunk, _sim) in enumerate(relevant_chunks, start=1):
        context_parts.append(f"[Фрагмент {i}]\n{chunk}")
    context = "\n\n".join(context_parts)

    system_text = (
        "Ты — помощник, который отвечает на вопросы по предоставленным фрагментам. "
        "Внимательно прочитай все фрагменты. Если в них упоминается объект из вопроса, "
        "используй это упоминание для ответа. Отвечай кратко и по делу, "
        "по возможности указывая номер фрагмента. "
        "Если информации в фрагментах действительно нет — скажи: "
        "«В предоставленных материалах ответа нет»."
    )

    return [
        {"role": "system", "content": system_text},
        {"role": "user", "content": f"Фрагменты:\n{context}\n\nВопрос: {question}"},
    ]


def ask_model(messages):
    """Отправляет промпт в чат-модель и возвращает ответ."""
    payload = {
        "model": CHAT_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": TEMPERATURE},
    }
    response = requests.post(CHAT_URL, json=payload, timeout=300)

    if response.status_code == 404:
        raise RuntimeError(
            f"Модель '{CHAT_MODEL}' не найдена в Ollama (404).\n"
            f"Проверьте: ollama list\n"
            f"Если имени нет, создайте псевдоним:\n"
            f"  ollama cp modelscope.cn/Qwen/Qwen2.5-3B-Instruct-GGUF:latest qwen2.5:3b"
        )

    response.raise_for_status()
    data = response.json()
    if "message" not in data or "content" not in data["message"]:
        raise RuntimeError(f"Неожиданный ответ от Ollama: {data}")
    return data["message"]["content"]


def main():
    try:
        chunks, vectors = load_index()
    except FileNotFoundError as e:
        print(f"Ошибка: {e}\nСначала: python build_index.py")
        return
    except RuntimeError as e:
        print(f"Ошибка: {e}")
        return

    print("RAG-чат по локальным документам. Введите вопрос или 'выход'.")
    print(f"(Модель: {CHAT_MODEL}, фрагментов: {len(chunks)}, TOP_K={TOP_K})\n")

    while True:
        question = input("Вы: ").strip()
        if question.lower() in ("выход", "exit", "quit"):
            print("До свидания!")
            break
        if not question:
            continue

        print("Ищу в документах...")
        try:
            relevant = find_relevant_chunks(question, chunks, vectors)
        except requests.exceptions.ConnectionError:
            print("Ollama не запущена. Запустите приложение Ollama.")
            continue
        except RuntimeError as e:
            print(f"Ошибка: {e}")
            continue

        print("\n--- Найденные фрагменты ---")
        for i, (chunk, sim) in enumerate(relevant, start=1):
            print(f"\n[{i}] сходство {sim:.3f}, длина {len(chunk)} символов:")
            print(chunk)

        print("\n--- Ответ модели ---")
        try:
            print(ask_model(build_prompt(question, relevant)))
        except requests.exceptions.ConnectionError:
            print("Ошибка соединения с Ollama.")
        except requests.exceptions.Timeout:
            print("Модель не ответила вовремя.")
        except RuntimeError as e:
            print(f"Ошибка: {e}")
        except requests.exceptions.HTTPError as e:
            print(f"HTTP-ошибка: {e}")
        print()


if __name__ == "__main__":
    main()
```

**Запуск:**

```powershell
python rag_chat.py
```

---

## Шаг 6. Тестирование — 7 вопросов по документу

После запуска `rag_chat.py` задайте вопросы по очереди. Ниже — что должно происходить с вашим документом про когнитивные искажения.

### Вопрос 1. Прямой по теме

```
Вы: Что такое катастрофизация?
```

**Ожидаемо:** топ-1 фрагмент — тот, где про катастрофизацию (сходство **0.7–0.85**), модель отвечает определением из текста, ссылается на фрагмент.

### Вопрос 2. Другой термин

```
Вы: Что такое эффект якоря?
```

**Ожидаемо:** топ-1 — фрагмент про якорь, сходство **0.7–0.85**. Модель пересказывает пример с ценой 10 000 → 5 000.

### Вопрос 3. Вопрос «перефраз»

```
Вы: Почему после новостей об авиакатастрофе люди боятся летать?
```

**Ожидаемо:** в топ-1 попадает фрагмент про **эвристику доступности** (сходство **0.6–0.75**), потому что там упомянут именно этот пример. Модель связывает вопрос с эвристикой доступности.

### Вопрос 4. Вопрос-«обобщение»

```
Вы: Какие искажения связаны с тревогой?
```

**Ожидаемо:** в топ-3 попадают фрагменты про катастрофизацию, чтение мыслей, возможно — про работу с искажениями. Модель перечисляет несколько.

### Вопрос 5. Вопрос вне документа

```
Вы: Как приготовить борщ?
```

**Ожидаемо:** сходство топ-1 **0.1–0.3**, ответ модели: «В предоставленных материалах ответа нет». Если модель начнёт «фантазировать» — уменьшите `TEMPERATURE` до `0.2`.

### Вопрос 6. Мусорный ввод

```
Вы: ghbdtn
```

**Ожидаемо:** то же самое — «нет в материалах». Если сходство окажется выше 0.5 — это тревожный сигнал: скорее всего, документ слишком короткий и неоднородный. С 12+ чанками такого не будет.

### Вопрос 7. Вопрос на понимание

```
Вы: Что такое ошибка выжившего и откуда взялся этот термин?
```

**Ожидаемо:** топ-1 — фрагмент про ошибку выжившего (сходство **0.75–0.85**). В ответе должно быть и определение, и пример с самолётами Второй мировой, и упоминание Абрахама Вальда.

### Как читать сходство

| Значение | Что означает |
| :--- | :--- |
| **0.7–0.9** | Отличное совпадение, фрагмент почти наверняка по теме |
| **0.5–0.7** | Хорошее совпадение, вероятно по теме |
| **0.3–0.5** | Слабое, «около темы» — норм для короткого индекса (2–3 чанка) |
| **< 0.3** | Не по теме |

---

## Шаг 7. Что делать, если модель говорит «нет ответа», а он есть

Порядок действий — от частого к редкому.

**1. Посмотрите полный фрагмент в консоли.** Скрипт печатает фрагмент целиком, без обрезки. Есть ли там ответ? Если нет — проблема в поиске, а не в модели.

**2. Проверьте сходство.** Если топ-1 < 0.5 — фрагмент подобран плохо. Причины:
- чанки слишком крупные, ответ «размазан» по нескольким;
- в вопросе лексика не пересекается с текстом;
- документ однородный (все абзацы про одно и то же).

**3. Увеличьте `TOP_K`** с 3 до 5. Иногда нужный фрагмент — второй или третий, а не первый.

**4. Переформулируйте вопрос ближе к тексту.** Эмбеддинги лучше работают при пересечении лексики: «Что говорится про ModelScope во фрагментах?» часто срабатывает там, где «ModelScope что такое?» — нет.

**5. Модель 3B слишком мала.** Qwen2.5-**3B** иногда отказывается даже при хорошем контексте — это ограничение размера. Обходные пути:
- скачать модель крупнее через ModelScope:
  ```powershell
  ollama pull modelscope.cn/Qwen/Qwen2.5-7B-Instruct-GGUF
  ollama cp modelscope.cn/Qwen/Qwen2.5-7B-Instruct-GGUF:latest qwen2.5:7b
  ```
  и в `rag_chat.py` заменить `CHAT_MODEL = "qwen2.5:3b"` на `"qwen2.5:7b"`. Размер ~4.7 ГБ, качество заметно выше;
- смягчить промпт: убрать из `system_text` фразу «Если информации действительно нет — скажи…», оставить только «опирайся на фрагменты».

**6. Уменьшите `TEMPERATURE` до 0.2**, если модель начинает «фантазировать» и добавлять факты, которых нет во фрагментах.

---

## Шаг 8. Что сдавать

```
13232/
├── documents.txt           ← исходный текст (15+ абзацев)
├── build_index.py          ← скрипт индексации
├── rag_chat.py             ← RAG-чат
├── chunks.json             ← создаётся build_index.py
├── embeddings.npy          ← создаётся build_index.py
└── README.md               ← полстраницы
```

**`README.md`** — полстраницы:
- тема и источник документа;
- команды запуска (`python build_index.py`, затем `python rag_chat.py`);
- 5 примеров вопросов с ответами и сходством (можно скриншотами терминала).

**Отчёт** — 1 страница:
- что получилось;
- где система ошибалась (нашла не тот фрагмент, сказала «не знаю»);
- как исправляли (переформулировали вопрос, увеличили `TOP_K`, расширили документ, сменили модель).

**Критерии** (для самопроверки):
- программа работает end-to-end — индекс строится, чат отвечает (40%);
- на тестовых вопросах находятся релевантные фрагменты (20%);
- ответы опираются на фрагменты, модель не «фантазирует» (20%);
- понятность кода и README (10%);
- рефлексия в отчёте (10%).

---

## Справка по API (актуально на ноябрь 2026)

| Что | Значение |
| :--- | :--- |
| **Эмбеддинги** | `POST http://localhost:11434/api/embed`. Поле запроса: `input` (строка или массив строк). Поле ответа: `embeddings` (массив векторов). |
| **Устаревший эндпоинт** | `/api/embeddings` (единственное число). Батчи не поддерживает. |
| **Чат** | `POST http://localhost:11434/api/chat`. Поле ответа: `message.content`. |
| **Модель эмбеддингов** | `nomic-embed-text` — 768 измерений, ~84 МБ в GGUF-квантовании. |
| **Зеркало для `ollama pull`** | `modelscope.cn/{owner}/{repo}` — работает, когда `registry.ollama.ai` недоступен. |
| **Псевдонимы** | `ollama cp <полное-имя> <короткое-имя>` — не копирует файлы, создаёт ссылку. |
| **Прокси для pip** | `$env:HTTP_PROXY = ""; $env:HTTPS_PROXY = ""` — если pip ругается на прокси. |
| **Прокси для Ollama** | Если прокси нужен для Ollama, задайте `$env:HTTPS_PROXY` в сессии, где запущен `ollama serve`. |

---

## Итог

Это руководство — финальная версия с:

- **Готовым `documents.txt`** (14 разделов про когнитивные искажения), проверенным на этой конфигурации;
- **Командами скачивания через ModelScope** (потому что `registry.ollama.ai` в вашей сети блокируется);
- **Созданием псевдонимов** `nomic-embed-text` и `qwen2.5:3b` через `ollama cp`;
- **Обходом проблемы с прокси** для `pip`;
- **Промптом с контекстом перед вопросом** и `temperature = 0.4` — вариант, который меньше «сдаётся» на Qwen2.5-3B;
- **Разбором 7 тестовых вопросов** и ожидаемых сходств;
- **Диагностикой 404** прямо в скрипте.

Если после расширения `documents.txt` до 12+ чанков сходство на «борщ» всё ещё > 0.4 — пришлите пример вывода, разберёмся, что в тексте «притягивает» нерелевантные вопросы.



## Промпт 1. Парсер документов (PDF → текст)

**Зачем:** сейчас вы вручную копируете текст из PDF в `documents.txt`. Это долго и чревато ошибками (переносы, артефакты вёрстки, потерянные абзацы). Промпт просит модель написать отдельный скрипт, который автоматически извлекает текст из PDF и сохраняет его в чистый `.txt`.

**Что важно знать:** для простых (не сканированных) PDF достаточно `pypdf` — это чистый Python-пакет без системных зависимостей, который на ноябрь 2026 года остаётся самым простым и надёжным выбором для базового извлечения текста. Для сканированных PDF (картинок) потребуется OCR, но это выходит за рамки проекта.

**Готовый промпт:**

```text
Ты — терпеливый преподаватель. Помоги мне написать скрипт на Python, который извлекает текст из PDF-файла и сохраняет его в текстовый файл documents.txt для дальнейшего использования в RAG-проекте.

Контекст: у меня есть папка data, в которой лежит книга или лекции в формате PDF (например, data/book.pdf). Мне нужно получить чистый текст без лишних артефактов, чтобы потом нарезать его на чанки.

Требования:
1. Используй библиотеку pypdf (актуальную на ноябрь 2026 года). Никаких PyMuPDF, pdfplumber или OCR — только pypdf для простоты.
2. Скрипт должен называться parse_pdf.py и лежать в папке code.
3. Путь к PDF-файлу задаётся константой в начале скрипта (например, PDF_PATH = "../data/book.pdf").
4. Скрипт должен:
   - открыть PDF через pypdf.PdfReader;
   - пройти по всем страницам и извлечь текст через page.extract_text();
   - склеить текст всех страниц с разделителем "\n\n";
   - сохранить результат в файл "../data/documents.txt" в кодировке UTF-8;
   - вывести в консоль количество страниц и общее количество извлечённых символов.
5. Обработай возможные ошибки:
   - если файл не найден — понятное сообщение;
   - если PDF зашифрован (нужен пароль) — сообщение, что файл защищён;
   - если страница не содержит текста (например, картинка) — пропустить её с предупреждением в консоли.
6. Комментарии в коде — на русском и короткие, для студентов-психологов, знающих только базовый Python.

Покажи полный код parse_pdf.py и кратко объясни, что делает каждая часть. Также напиши команду установки библиотеки в PowerShell.
```

**Что этот промпт тренирует:** работу с файлами, обработку ошибок, понимание того, что PDF — это не просто «текст в коробочке», а структура, из которой текст ещё нужно извлечь.

---

## Промпт 2. Улучшить чанкинг (переход к токенам и рекурсивному разбиению)

**Зачем:** сейчас чанки режутся «по символам» — это грубо. Модель эмбеддингов `nomic-embed-text` работает с токенами, а не с символами. Бенчмарки 2026 года показывают, что **рекурсивное разбиение по 512 токенов с перекрытием 50–100 токенов** — это проверенный дефолт для большинства RAG-приложений, который на реальных документах обгоняет более сложные методы. Для учебного проекта достаточно начать с 512 токенов и перекрытия 10–15%.

**Готовый промпт:**

```text
Перепиши функцию split_into_chunks в моём build_index.py так, чтобы она резала текст не по символам, а по токенам.

Требования:
1. Используй библиотеку tiktoken (pip install tiktoken) с кодировкой cl100k_base — это стандартный токенизатор, который хорошо работает как прокси для большинства моделей. Если tiktoken недоступен, можно использовать простую оценку: 1 токен ≈ 4 символа, но предпочтительнее tiktoken.
2. Размер чанка: 512 токенов. Перекрытие: 64 токена (около 12.5%).
3. Реализуй рекурсивное разбиение: сначала пытайся резать по абзацам ("\n\n"), затем по предложениям ("."), и только в крайнем случае — по токенам. Это сохраняет смысловую целостность кусков.
4. Если после разбиения получается кусок короче 50 токенов, присоедини его к предыдущему (чтобы не было «огрызков»).
5. Функция должна принимать текст и возвращать список строк (чанков).
6. Покажи полный обновлённый build_index.py с этой функцией и краткими комментариями на русском.

Вот мой текущий build_index.py:
[вставьте код build_index.py]
```

**Что этот промпт тренирует:** понимание разницы между символами и токенами; знакомство с рекурсивным разбиением как компромиссом между простотой и качеством.

---

## Промпт 3. Гибридный поиск: добавить BM25 к векторному поиску

**Зачем:** векторный поиск хорошо находит «по смыслу», но иногда промахивается, когда в вопросе есть точный термин или редкое слово. BM25 (классический поиск по ключевым словам) страхует в таких случаях. Исследования 2026 года показывают, что гибридный поиск (BM25 + векторы) с объединением результатов через **Reciprocal Rank Fusion (RRF)** даёт заметный прирост: например, MRR@3 с 0.74 до 0.79, nDCG@3 с 0.76 до 0.81. Это не «магия», а понятная комбинация двух сигналов: лексического и семантического.

**Готовый промпт:**

```text
Улучши мой rag_chat.py: добавь гибридный поиск, который объединяет векторный поиск (косинусное сходство эмбеддингов) и поиск по ключевым словам BM25.

Требования:
1. Используй библиотеку rank_bm25 (pip install rank_bm25) — она простая и не требует тяжёлых зависимостей.
2. При старте программы построй BM25-индекс по всем чанкам (токенизируй текст простым split() по пробелам и приведи к нижнему регистру).
3. Для каждого вопроса:
   - найди top-10 чанков векторным поиском (как сейчас);
   - найди top-10 чанков через BM25;
   - объедини два списка через Reciprocal Rank Fusion (RRF) с параметром k=60 (стандартное значение). RRF: score(chunk) = сумма по каждому списку 1 / (k + rank).
   - выбери top-3 чанка по итоговому RRF-скору.
4. В консоли показывай, какие чанки пришли из векторного поиска, какие из BM25, и итоговый список после RRF — это поможет понять, что происходит.
5. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что «поиск» — это не одна волшебная кнопка, а комбинация сигналов; знакомство с RRF как простым и эффективным способом объединения ранжирований.

---

## Промпт 4. Стриминг ответа в терминале

**Зачем:** сейчас `rag_chat.py` ждёт весь ответ целиком и только потом печатает. В первом проекте вы уже делали стриминг для обычного чата — теперь нужно адаптировать его для RAG. Важно: Ollama использует **NDJSON** (Newline-Delimited JSON), а не Server-Sent Events. Каждая строка — самостоятельный JSON-объект, и его нужно разбирать через `json.loads` построчно.

**Готовый промпт:**

```text
Перепиши функцию ask_model и цикл вывода в моём rag_chat.py так, чтобы ответ модели печатался в терминал постепенно (стриминг), а не одним куском.

Требования:
1. Используй эндпоинт /api/chat с параметром stream=True.
2. Ollama присылает NDJSON: каждая строка — отдельный JSON-объект. Разбирай ответ построчно через response.iter_lines():
   - для каждой непустой строки декодируй её (utf-8) и разбери через json.loads;
   - бери поле chunk["message"]["content"] (если оно есть);
   - печатай этот кусочек через print(..., end="", flush=True).
3. После окончания стриминга добавь перевод строки.
4. Обработай ошибки: ConnectionError, HTTPError, Timeout — с понятными сообщениями, как в текущем коде.
5. Сохрани всю текущую логику: поиск фрагментов, показ найденных фрагментов перед ответом.
6. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** работу с потоковыми ответами; понимание того, что формат ответа при `stream=True` отличается от обычного JSON.

---

## Промпт 5. Улучшить системный промпт (заставить модель ссылаться на фрагменты)

**Зачем:** даже с RAG модель иногда «забывает» про фрагменты и отвечает «из головы». Системный промпт — главный инструмент управления. Нужно явно потребовать: отвечать только по фрагментам, указывать номер фрагмента, честно говорить «не знаю», если ответа нет.

**Готовый промпт:**

```text
Перепиши системный промпт в функции build_prompt моего rag_chat.py так, чтобы модель строго следовала правилам RAG.

Требования к новому системному промпту:
1. Роль: «Ты — помощник по психологии, который отвечает на вопросы, опираясь ТОЛЬКО на предоставленные фрагменты текста».
2. Обязательное правило: «Если в фрагментах нет прямого ответа на вопрос, скажи: "В предоставленных материалах ответа нет" и не пытайся угадать».
3. Требование ссылаться на источник: «В конце каждого утверждения указывай номер фрагмента в квадратных скобках, например: [Фрагмент 2]».
4. Запрет на выдумывание: «Не добавляй факты, которых нет во фрагментах, даже если ты уверен, что они верны».
5. Краткость: «Отвечай по существу, без вступлений вроде "Конечно!" или "Отличный вопрос!"».

Покажи полный обновлённый rag_chat.py с новым системным промптом. Комментарии — на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что системный промпт — это «инструкция по эксплуатации» модели, а не просто вежливая фраза.

---

## Промпт 6. Добавить порог отсечения (не подставлять «мусор»)

**Зачем:** сейчас в промпт всегда попадают top-3 чанка, даже если они совсем нерелевантны. Если вопрос не по теме документов, модель получит «мусор» и может ответить ерунду. Решение — ввести порог косинусного сходства: если лучший чанк ниже порога, не подставлять его вообще и честно сказать пользователю, что в материалах нет ответа.

**Готовый промпт:**

```text
Добавь в rag_chat.py порог отсечения для найденных фрагментов.

Требования:
1. Задай константу MIN_SIMILARITY = 0.35 в начале файла (подбери экспериментально: для nomic-embed-text хорошие результаты обычно имеют сходство 0.5–0.7, «мусор» — ниже 0.3).
2. В функции find_relevant_chunks после сортировки отбрось все чанки, у которых сходство ниже MIN_SIMILARITY.
3. Если после отсечения не осталось ни одного чанка, в main() не отправляй запрос в модель, а выведи: «В предоставленных материалах нет информации по этому вопросу. Попробуйте переформулировать или задать другой вопрос».
4. Если осталось меньше TOP_K чанков — используй столько, сколько есть.
5. Покажи полный обновлённый rag_chat.py с комментариями на русском.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]
```

**Что этот промпт тренирует:** понимание того, что RAG — это не «всегда отвечай», а «отвечай, когда есть на что опереться».

---

## Промпт 7. Сохранение и загрузка индекса без пересчёта

**Зачем:** сейчас `build_index.py` при каждом запуске пересчитывает эмбеддинги для всех чанков. Если документы не менялись, это трата времени. Нужно проверять, существует ли уже готовый индекс, и загружать его.

**Готовый промпт:**

```text
Улучши build_index.py так, чтобы он не пересчитывал эмбеддинги, если индекс уже существует и документы не менялись.

Требования:
1. При запуске проверяй, существуют ли файлы chunks.json и embeddings.npy.
2. Если оба файла существуют, выведи: «Индекс уже построен. Если вы изменили документы, удалите chunks.json и embeddings.npy и запустите снова». И заверши работу без пересчёта.
3. Добавь флаг --force (или переменную FORCE_REBUILD = False в начале файла), который позволяет принудительно перестроить индекс, даже если файлы есть.
4. Покажи полный обновлённый build_index.py с комментариями.

Вот мой текущий build_index.py:
[вставьте код build_index.py]
```

**Что этот промпт тренирует:** понимание жизненного цикла индекса; аккуратность при работе с файлами.

---

## Промпт 8. Автоматическая оценка качества поиска

**Зачем:** чтобы понять, стало ли лучше после изменений, нужны метрики. Простейший способ — задать 10 тестовых вопросов с заранее известными правильными ответами (или хотя бы «правильными» фрагментами) и посчитать, на каком месте (rank) находится нужный фрагмент. Это учит не «верить на глаз», а измерять.

**Готовый промпт:**

```text
Добавь в проект отдельный скрипт evaluate.py, который оценивает качество поиска на наборе тестовых вопросов.

Требования:
1. Создай файл test_questions.json со списком из 10 вопросов. Для каждого вопроса укажи:
   - question: текст вопроса;
   - expected_keywords: список из 1–3 ключевых слов, которые должны быть в правильном чанке (например, для вопроса «Что такое катастрофизация?» — ["катастрофизация", "преувеличение"]).
2. Скрипт evaluate.py:
   - загружает chunks.json, embeddings.npy и test_questions.json;
   - для каждого вопроса находит top-5 чанков (тем же способом, что в rag_chat.py);
   - проверяет, содержит ли хотя бы один из top-5 чанков все expected_keywords;
   - считает, на каком месте (1–5) впервые встречается «правильный» чанк;
   - выводит таблицу: вопрос | найден ли правильный чанк | на каком месте;
   - в конце выводит итоговую метрику: долю вопросов, где правильный чанк попал в top-3 (это простой аналог Recall@3).
3. Используй только стандартные библиотеки + numpy + requests (как в остальном проекте).
4. Покажи полный код evaluate.py и пример test_questions.json с 3 вопросами для демонстрации.

Контекст: вот мой текущий find_relevant_chunks из rag_chat.py:
[вставьте функцию find_relevant_chunks или весь rag_chat.py]
```

**Что этот промпт тренирует:** мышление в терминах метрик; понимание того, что «улучшение» — это не субъективное ощущение, а измеримая величина.

---

## Как пользоваться этими промптами (памятка)

1. **Выберите один промпт**, скопируйте его в DeepSeek или ChatGPT.
2. **Приложите свой текущий код** (полный файл или указанную функцию).
3. Получите обновлённый код. **Сравните** с исходным: что именно изменилось? Прочитайте комментарии.
4. **Запустите** и проверьте, что улучшение действительно работает (например, для промпта 8 — сравните метрику до и после).
5. **Сохраните копию** перед изменениями: `rag_chat_v1.py`, `rag_chat_v2.py` — чтобы можно было откатиться.

Эти промпты покрывают основные «болевые точки» RAG-проекта: качество парсинга, качество чанкинга, качество поиска, честность модели и измеримость результата. Начинать логично с **Промпта 1** (парсер), затем **Промпта 2** (чанкинг), затем **Промпта 5** (системный промпт) — это даст наибольший эффект при минимальных изменениях.

## Streamlit-интерфейс



## Промпт 1. Базовый Streamlit-интерфейс для RAG-чата

**Что даёт:** рабочее приложение `app.py`, где можно писать вопрос, видеть ответ, видеть найденные фрагменты со сходством, менять `TOP_K` и `TEMPERATURE` в сайдбаре и выбирать чат-модель из списка Ollama. Логика берётся из твоего `rag_chat.py`.

```text
Ты — терпеливый преподаватель Python. Помоги превратить мой консольный RAG-чат в веб-приложение на Streamlit. Мне нужен рабочий каркас, который я потом буду украшать.

Контекст проекта:
- RAG по локальному документу documents.txt.
- Эмбеддинги и чат — через локальную Ollama (http://localhost:11434).
- Индекс уже построен: chunks.json и embeddings.npy (размерность 768).
- Модель эмбеддингов: nomic-embed-text.
- Чат-модели доступны как псевдонимы: qwen2.5:3b, qwen2.5:7b (могут быть и другие).

Требования:
1. Создай файл app.py в корне проекта (рядом с chunks.json и embeddings.npy).
2. Используй streamlit (pip install streamlit). Никаких Flask/FastAPI — только Streamlit.
3. Функции из моего rag_chat.py (load_index, get_embedding, cosine_similarity, find_relevant_chunks, build_prompt, ask_model) перенеси в отдельный модуль rag_core.py без print() и без input(), чтобы их можно было импортировать. В app.py — только UI.
4. Сайдбар:
   - Заголовок «Настройки».
   - Выпадающий список моделей чата. Список получай автоматически через GET http://localhost:11434/api/tags (поле models[].name), отфильтруй модели эмбеддингов (в имени есть "embed") и покажи пользователю. Если запрос не удался — fallback на ["qwen2.5:3b", "qwen2.5:7b"].
   - Слайдер TOP_K от 1 до 8, по умолчанию 3.
   - Слайдер TEMPERATURE от 0.0 до 1.0, шаг 0.05, по умолчанию 0.4.
   - Чекбокс «Показывать найденные фрагменты», по умолчанию включён.
   - Кнопка «Очистить историю чата».
5. Основная область:
   - Заголовок «RAG-ассистент по документам».
   - История чата через st.chat_message("user") / st.chat_message("assistant") и st.chat_input.
   - История хранится в st.session_state.messages (список словарей role/content). В session_state также храни фрагменты последнего ответа, чтобы они не пропадали при перерисовке.
6. При новом вопросе:
   - найди top-K фрагментов (как в rag_chat.py, но с параметрами из сайдбара);
   - построй промпт через build_prompt;
   - вызови Ollama /api/chat с stream=False (стриминг сделаем отдельным промптом);
   - покажи ответ;
   - под ответом в st.expander("Источники (фрагменты)") выведи каждый фрагмент: номер, сходство (3 знака после запятой), длину, сам текст в st.text_area или st.markdown с ограничением высоты.
7. Обработка ошибок — обязательно и по-человечески:
   - Ollama недоступна (requests.exceptions.ConnectionError) → st.error("Ollama не запущена. Запустите приложение Ollama и повторите.").
   - Файлы chunks.json / embeddings.npy не найдены → st.warning("Индекс не найден. Сначала запустите: python build_index.py").
   - HTTP 404 на модель → st.error с подсказкой про ollama cp и ollama list.
   - Пустой ввод — игнорировать.
8. В конце файла: подсказка в st.caption, как запустить: `streamlit run app.py`.
9. Код — с короткими комментариями на русском, для студента-психолога с базовым Python. Без ООП, без декораторов, кроме @st.cache_data для load_index (индекс грузится один раз).

Покажи:
- полный rag_core.py;
- полный app.py;
- команду установки streamlit и команду запуска.

Вот мой текущий rag_chat.py:
[вставьте код rag_chat.py]

Вот мой текущий build_index.py (для понимания формата chunks.json и embeddings.npy):
[вставьте код build_index.py]
```

---

## Промпт 2. «Сделай красиво»: кастомный CSS, темы, карточки, анимации

**Что даёт:** тот же функционал, но с приятным глазу оформлением: свои цвета, скруглённые карточки источников, аватары, аккуратные отступы, спиннеры, подсветка сходства (зелёный/жёлтый/красный), аккуратный заголовок, поддержка тёмной и светлой темы.

```text
Ты — фронтенд-дизайнер и Python-разработчик. У меня есть рабочее Streamlit-приложение app.py для RAG-чата. Нужно сделать его визуально приятным, современным и аккуратным, не ломая логику.

Что должно остаться без изменений:
- Вся логика поиска и вызова Ollama (импорт из rag_core.py).
- session_state.messages и session_state.last_sources.
- Выбор модели, TOP_K, TEMPERATURE в сайдбаре.
- Показ источников в раскрывающемся блоке.

Что нужно добавить:
1. Кастомный CSS через st.markdown(..., unsafe_allow_html=True):
   - шрифт Inter (или system-ui, если Inter недоступен) для всего текста;
   - мягкий фон страницы (светлая тема: #f7f8fb, тёмная: #0e1117) — использовать встроенную поддержку тем Streamlit, но подкрасить;
   - скругления у карточек и полей ввода (border-radius: 14px);
   - тень у карточек источников (box-shadow: 0 2px 12px rgba(0,0,0,0.06));
   - аккуратные отступы между сообщениями чата;
   - стилизованный заголовок приложения с градиентом текста.
2. Красивый заголовок: эмодзи + название + короткое описание под ним через st.caption.
3. Аватары для st.chat_message: пользователь — "🧑‍💻", ассистент — "🧠".
4. Спиннер при поиске и при обращении к модели: with st.spinner("Ищу в документах..."): ... и with st.spinner("Думаю..."): ...
5. Карточки источников: вместо «голого» текста — каждая в отдельном контейнере st.container(border=True), внутри:
   - строка «Фрагмент N · сходство 0.78 · 583 символа»;
   - цветной бейдж сходства: зелёный (>=0.7), жёлтый (0.5–0.7), красный (<0.5), цвет через inline-HTML span;
   - сам текст фрагмента с ограничением высоты (max-height: 220px; overflow-y: auto).
6. Прогресс-бар сходства: st.progress(min(sim, 1.0)) под бейджем.
7. Пустое состояние (когда нет сообщений): приветственный блок с 3–4 кнопками-примерами вопросов (например, «Что такое катастрофизация?», «Что такое эффект якоря?», «Почему люди боятся летать после новостей об авиакатастрофе?»). При нажатии — вопрос отправляется в чат.
8. Иконки для кнопок (эмодзи) и аккуратный сайдбар: заголовок «⚙️ Настройки», тонкие разделители st.divider().
9. Подвал страницы: маленькая подпись «Работает локально через Ollama · модель: {выбранная} · TOP_K={k}».
10. Совместимость со светлой и тёмной темой Streamlit (не хардкодить цвета текста, кроме бейджей; бейджи — с полупрозрачным фоном).

Ограничения:
- Только Streamlit + стандартные CSS-вставки. Никаких внешних JS-фреймворков.
- Не ломать существующие ключи session_state.
- Код с комментариями на русском.

Покажи полный обновлённый app.py. Если для бейджей нужна вспомогательная функция (например, render_similarity_badge(sim)), вынеси её отдельно с комментарием.

Вот мой текущий app.py:
[вставьте код app.py из Промпта 1]

Вот мой rag_core.py:
[вставьте код rag_core.py]
```

---

## Промпт 3. «Всё в одном»: вкладки — Чат / Модели / Индекс / Оценка

**Что даёт:** многопанельное приложение. Вкладка «Чат» — как раньше. Вкладка «Модели» — список моделей Ollama, их размеры, кнопки `ollama pull` (через зеркало ModelScope), `ollama cp` для псевдонимов, удаление. Вкладка «Индекс» — кнопка пересборки, статус файлов, число чанков и размерность. Вкладка «Оценка» — прогон `test_questions.json`, таблица результатов, Recall@3. Плюс стриминг ответа — чтобы было живо.

**Почему это удобно:** не нужно прыгать в терминал ради `ollama pull` или пересборки индекса — всё в одном окне.

```text
Ты — старший Python-разработчик. У меня есть Streamlit-приложение app.py для RAG-чата. Нужно расширить его до многопанельного «пульта управления» с вкладками, сохранив существующий функционал и стиль.

Стек: streamlit, requests, numpy, pandas, subprocess (для вызова ollama), json. Ничего лишнего.

Структура приложения (st.tabs):
1. Вкладка «💬 Чат» — текущий RAG-чат:
   - история в session_state.messages;
   - выбор модели в сайдбаре;
   - TOP_K, TEMPERATURE в сайдбаре;
   - карточки источников в st.expander;
   - стриминг ответа: /api/chat с stream=True, NDJSON, iter_lines(), json.loads, поле chunk["message"]["content"], печать через placeholder.markdown(..., end="") — обновляй placeholder по мере прихода кусочков.
2. Вкладка «🧠 Модели»:
   - таблица всех моделей из GET /api/tags: имя, размер (МБ/ГБ), дата изменения — через pandas.DataFrame и st.dataframe.
   - Кнопка «Обновить список».
   - Поле «Скачать модель через ModelScope»: пользователь вводит, например, modelscope.cn/Qwen/Qwen2.5-7B-Instruct-GGUF — кнопка запускает subprocess.run(["ollama", "pull", имя], capture_output=True, text=True), вывод показывается в st.code / st.text_area. Предупреди, что скачивание может быть долгим.
   - Поле «Создать псевдоним»: два текстовых поля (полное имя, короткое имя), кнопка → subprocess.run(["ollama", "cp", full, short]). После успеха — st.success и обновление таблицы.
   - Кнопка «Удалить модель» с подтверждением через st.checkbox «Я уверен» → subprocess.run(["ollama", "rm", имя]).
   - Все subprocess-вызовы оборачивай в try/except, при ошибке — st.error с текстом ошибки из stderr.
3. Вкладка «📚 Индекс»:
   - статус: существуют ли chunks.json и embeddings.npy, их размеры и дата изменения.
   - если существуют — покажи число чанков (len(json.load(chunks.json))) и форму embeddings.npy (np.load(...).shape).
   - кнопка «Пересобрать индекс»: запускает subprocess.run(["python", "build_index.py"], capture_output=True, text=True), показывает stdout/stderr в st.code.
   - чекбокс «Принудительно» (для будущего флага --force) — пока просто прокидывай как аргумент, если он есть.
   - предупреждение, если documents.txt новее, чем embeddings.npy (сравнивай os.path.getmtime) — «Документ изменён, возможно, индекс устарел».
4. Вкладка «📊 Оценка»:
   - загружает test_questions.json (формат: список объектов {question, expected_keywords});
   - кнопка «Прогнать оценку»: для каждого вопроса находит top-5 через find_relevant_chunks, проверяет, есть ли в чанке все expected_keywords (регистронезависимо), определяет первое место попадания;
   - результат — pandas.DataFrame с колонками: Вопрос | Найдено | Место (1–5) | Сходство top-1;
   - итог: Recall@3 в процентах, крупным шрифтом через st.metric("Recall@3", f"{x}%").
   - если test_questions.json нет — st.info с шаблоном: покажи пример JSON в st.code.

Общие требования:
1. Сайдбар общий для всех вкладок: выбор модели, TOP_K, TEMPERATURE — только на вкладке «Чат» они реально используются, но пусть будут видны всегда (с подписью «Применяется на вкладке Чат»).
2. Все длительные операции (pull, rm, cp, пересборка индекса, оценка) — с st.spinner и понятными сообщениями.
3. Никаких падений при отсутствии Ollama: если /api/tags недоступен, все вкладки показывают мягкое предупреждение, а «Чат» — st.error и блокируется ввод.
4. Логика поиска/промпта/стриминга — в rag_core.py. app.py — только UI и вызовы subprocess.
5. Сохраняй красивое оформление из предыдущего шага: заголовок с градиентом, карточки источников, бейджи сходства (зелёный/жёлтый/красный), пустое состояние с кнопками-примерами.
6. Комментарии на русском, короткие. Без ООП, максимум — вспомогательные функции.
7. В конце файла — подсказка, как запустить: `streamlit run app.py`.

Покажи:
- полный rag_core.py (с добавленной функцией get_ollama_models());
- полный app.py;
- пример test_questions.json с 3 вопросами под документ про когнитивные искажения.

Вот мой текущий app.py (после шагов 1–2):
[вставьте код app.py]

Вот rag_core.py:
[вставьте код rag_core.py]

Вот мой build_index.py:
[вставьте код build_index.py]
```

---

## Как этим пользоваться (коротко)

1. **Сначала Промпт 1.** Получи рабочий `app.py` + `rag_core.py`. Запусти `streamlit run app.py`, проверь, что чат отвечает и источники видны.
2. **Потом Промпт 2.** Наложи красоту. Проверь, что не сломались `session_state` и показ источников, что в тёмной и светлой теме всё читается.
3. **В конце Промпт 3.** Добавь вкладки. Это самый большой шаг — не стесняйся запускать по частям: сначала вкладка «Модели», потом «Индекс», потом «Оценка».

Что важно проверить самому после каждого шага:
- `streamlit run app.py` запускается без ошибок в консоли;
- при выключенной Ollama приложение не падает, а показывает понятное сообщение;
- в источниках сходство и текст совпадают с тем, что ты видел в консольном `rag_chat.py`;
- выбор модели в сайдбаре действительно меняет модель (проверь на `qwen2.5:7b`, если скачал).

Если после Промпта 3 что-то отвалится — самый частый виновник `subprocess` и пути к `build_index.py` / `test_questions.json`. Тогда просто напиши в ответ: «subprocess не находит файл», и я подскажу, как поправить (обычно `cwd=os.path.dirname(__file__)` или `sys.executable` вместо `"python"`).